In [1]:
from pathlib import Path
from tqdm import tqdm

import os, re, math
from PIL import Image, ImageDraw, ImageFont

In [2]:
model_name = 'DMFC-FAS'  # MVP-FAS | DMFC-FAS | CVPR2024-FAS
domain_name = 'mask'  # cutout | figure | mannequin | mask | print | screen

path = Path(model_name) / domain_name / 'examples'
path_res = Path(model_name) / domain_name / 'progress_imgs.png'

path

WindowsPath('DMFC-FAS/mask/examples')

In [13]:
# --- НАСТРОЙКИ ---
cols = 5          # Кол-во колонок
scale = 0.25       # Коэффициент уменьшения картинок (0.5 = в 2 раза меньше)
gap_x = 80        # Отступ по горизонтали
gap_y = 60        # Отступ по вертикали
txt_h = 40        # Высота зоны под текст
font_size = 16    # Размер шрифта
arrow_width = 6   # Толщина линии стрелки
# -----------------

In [14]:
list_filenames = sorted(
    [f for f in os.listdir(path) if f.endswith(".png")],
    key=lambda f: int(re.search(r"step_(\d+)", f).group(1))
)
print(list_filenames[:5])

['step_0_score=0.78932.png', 'step_2_score=0.79392.png', 'step_7_score=0.83049.png', 'step_42_score=0.84138.png', 'step_45_score=0.88579.png']


In [15]:
imgs_raw = [Image.open(os.path.join(path, f)).convert("RGB") for f in list_filenames]
orig_w, orig_h = imgs_raw[0].size

# Уменьшаем картинки
new_w = int(orig_w * scale)
new_h = int(orig_h * scale)
imgs = [img.resize((new_w, new_h), Image.LANCZOS) for img in imgs_raw]

rows = math.ceil(len(imgs) / cols)
out_w = cols * new_w + (cols - 1) * gap_x
out_h = rows * (new_h + txt_h) + (rows - 1) * gap_y + 20
out = Image.new("RGB", (out_w, out_h), "white")
draw = ImageDraw.Draw(out)

In [16]:
# Попытка загрузить красивый шрифт, иначе дефолтный
try:
    font = ImageFont.truetype("arial.ttf", font_size)
except IOError:
    font = ImageFont.load_default()

In [17]:

for i, (img, name) in enumerate(zip(imgs, list_filenames)):
    row, col = divmod(i, cols)
    x = col * (new_w + gap_x)
    y = row * (new_h + txt_h + gap_y)

    # Рисуем текст
    name = name[:-4]
    name = name.split('_')
    name = f'{name[0]}={name[1]} | {name[2]}'
    draw.text((x, y + 5), name, fill="black", font=font)
    
    # Вставляем картинку
    out.paste(img, (x, y + txt_h))

    # Рисуем стрелку, если это не последняя картинка в строке и не самая последняя вообще
    if col < cols - 1 and i < len(imgs) - 1:
        mid_y = y + txt_h + new_h // 2
        start_x = x + new_w + 10
        end_x = x + new_w + gap_x - 10
        
        # Линия
        draw.line([(start_x, mid_y), (end_x, mid_y)], fill="black", width=arrow_width)
        
        # Наконечник стрелки (треугольник)
        arrow_head_size = 12
        draw.polygon([
            (end_x, mid_y), 
            (end_x - arrow_head_size, mid_y - arrow_head_size//2), 
            (end_x - arrow_head_size, mid_y + arrow_head_size//2)
        ], fill="black")



out.save(path_res)
print('ready')

ready
